# Introducción a la Ingeniería de Datos

## Trabajo Práctico N.º 2
### Modelado de datos para un medio internacional de noticias: base operativa (OLTP) y almacén dimensional (OLAP) sobre GDELT 2.0 / CAMEO

**Autor:** Matías R. Pretel

> **Diagramas:** `TP2.drawio` (pestaña 1: modelo OLTP Entidad-Relación; pestaña 2: modelo dimensional en estrella).

## Instrucciones de la actividad

**Trabajo práctico 2**

**IMPORTANTE:** Al utilizar herramientas de inteligencia artificial deben indicar qué IA están usando y con qué propósito. Por ejemplo: "Claude Opus 4.6" para "mejorar redacción de textos" o "Chat GPT 5.6" para "crear código fuente".

En el TP1 procesaron el flujo de datos de GDELT 2.0. Ahora, la dirección de un Medio Internacional de Noticias necesita modernizar su plataforma de datos para dos propósitos clave:

- **Operación Periodística (OLTP):** Registrar en tiempo real las noticias publicadas, las fuentes consultadas, los actores involucrados y el seguimiento de eventos en desarrollo.
- **Inteligencia de Contenidos (OLAP):** Analizar el impacto, la cobertura global y las tendencias mediáticas para tomar decisiones editoriales y de distribución en tiempo real.

**Recursos:**

- Lista de URLs con actualizaciones cada 15 minutos https://data.gdeltproject.org/gdeltv2/masterfilelist.txt .
- URL oficial del libro de códigos de GDELT 2.0 https://data.gdeltproject.org/documentation/GDELT-Event_Codebook-V2.0.pdf
- URL de la documentación en texto/HTML donde se enuncian los campos https://gdeltproject.org/data/documentation/CAMEO.Manual.1.1b3.pdf

**Requisitos:**

Su diseño del Almacén de Datos debe estar estructurado de forma tal que permita responder estas 5 preguntas clave del negocio periodístico:

1. **Rendimiento e Impacto:** ¿Cuáles son las noticias y temas que generan el mayor volumen de menciones y publicaciones (NumMentions / NumArticles) por país y mes?
2. **Sentimiento de la Cobertura:** ¿Cómo varía el tono promedio de la cobertura (AvgTone) sobre eventos globales según el tipo de actor involucrado (ej. Gobierno, ONG, Empresas)?
3. **Análisis de Frecuencia y Temporalidad:** ¿Cómo cambia la intensidad del flujo de noticias entre días hábiles, fines de semana y eventos festivos?
4. **Cobertura Multi-Temática (Múltiples Categorías CAMEO):** ¿Qué eventos periodísticos requirieron clasificación bajo múltiples códigos de acción o temáticas simultáneas en una misma cobertura?

**Actividades:**

1) Diseñe la estructura de la base de datos operativa basándose en el estándar CAMEO/GDELT.
   - Presente un diagrama Entidad-Relación. Presente las entidades principales (Noticia_Evento, Actor, Ubicación, Fuente_Medio, Código_CAMEO).
   - Identifique Llaves: PK, FK, Llave Compuesta, Llave Sustituta (Surrogate) justificando su uso frente a la llave natural (GlobalEventID).
   - Indique el tipo de relaciones: 1:1, 1:N, M:N, opcional, obligatoria.

2) Diseñe el modelo dimensional para la tabla de hechos del análisis de medios.
   - Esquema Estrella: Dibuje la tabla de hechos central y sus dimensiones asociadas. Indique las métricas en la tabla de hechos.
   - Dimensión Temporal: Modele la dimensión de tiempo con los atributos: Año, Trimestre, Mes, Día_Semana, Es_Fin_De_Semana y Es_Festivo.
   - Tabla Puente (Bridge Table): Diseñe e implemente una Tabla Puente para resolver la relación de una noticia/evento con múltiples códigos de categorías temáticas (CAMEO).

3) Opcional, implemente ambas bases de datos (transaccional y almacén) mediante consultas SQL. Puede usar python, SQLite o PostgreSQL. Proporcione el código ejecutable para reproducir sus modelos.

**Formato de entrega:** Archivo de Draw.io en formato XML con el nombre "TP2" dentro de su repositorio Github. En esta actividad deben pegar la URL con el acceso al archivo de diagrama.

### Uso de inteligencia artificial

Se utilizó **Claude Opus 5.5** (Anthropic) con los siguientes propósitos:

- Obtener los campos y atributos de las entidades de la base de datos a partir de la documentación de GDELT 2.0 y CAMEO.
- Hacer *brainstorming* sobre el diseño de los modelos.
- Generar una primera aproximación de los diagramas en draw.io, que luego fueron revisados y simplificados manualmente.

## Actividad 1: Modelo operativo (OLTP)

Referencia: `TP2.drawio`, **pestaña 1** (*Modelo OLTP (Entidad-Relación)*). El diagrama está simplificado a propósito para que sea legible.

### 1.1 Entidades

**Tabla 1.** Entidades del modelo operativo.

| Entidad | Descripción | Tipo |
|---|---|---|
| `NOTICIA_EVENTO` | Un evento/noticia y sus métricas de cobertura | Principal |
| `ACTOR` | Persona, organización o país que participa en eventos | Principal |
| `UBICACION` | Lugar geográfico (del actor o de la acción) | Principal |
| `FUENTE_MEDIO` | Medio que publica artículos | Principal |
| `CODIGO_CAMEO` | Catálogo jerárquico de códigos de acción (raíz → base → evento) | Principal |
| `PAIS` | País, con código FIPS (geografía) y CAMEO (actores) | Catálogo |
| `TIPO_ACTOR` | Tipos CAMEO de actor (`GOV`, `NGO`, `BUS`, …) y su categoría | Catálogo |
| `EVENTO_ACTOR` | Participación de un actor en un evento con un rol (1 = Actor1, 2 = Actor2) | Asociativa |
| `EVENTO_CATEGORIA` | Clasificación de un evento bajo uno o varios códigos CAMEO | Asociativa |
| `MENCION` | Cada artículo que menciona el evento (seguimiento de eventos en desarrollo) | Asociativa |
| `ACTOR_TIPO` | Hasta tres tipos por actor, ordenados | Asociativa |
| `ARTICULO_ORIGEN` | Primer artículo que reportó el evento | Dependiente (1:1) |

`NOTICIA_EVENTO`, `ACTOR`, `UBICACION`, `FUENTE_MEDIO` y `CODIGO_CAMEO` son las cinco entidades principales. Las demás existen porque `PAIS` y `TIPO_ACTOR` evitan repetir códigos y descripciones, las asociativas resuelven relaciones M:N y `ARTICULO_ORIGEN` separa la URL de origen en una relación 1:1.

### 1.2 Atributos

**Tabla 2.** Atributos de las entidades principales (entre paréntesis, el campo de GDELT 2.0 de origen).

| Entidad | Atributo | Llave | Descripción |
|---|---|---|---|
| `NOTICIA_EVENTO` | `evento_id` | PK | Identificador interno del evento (sustituta) |
| | `global_event_id` | UK | Identificador de GDELT (`GLOBALEVENTID`); NULL en noticias propias del medio |
| | `ubicacion_accion_id` | FK | Lugar donde ocurre la acción (`ActionGeo_*`) |
| | `fecha_evento` | | Fecha en que ocurrió el evento (`SQLDATE`) |
| | `fecha_agregado` | | Fecha y hora en que GDELT registró el evento (`DATEADDED`) |
| | `es_evento_raiz` | | Si el evento aparece en el párrafo principal del artículo (`IsRootEvent`) |
| | `quad_class` | | Clase general: cooperación verbal/material, conflicto verbal/material (`QuadClass`) |
| | `escala_goldstein` | | Impacto teórico sobre la estabilidad, de -10 a +10 (`GoldsteinScale`) |
| | `num_menciones` | | Total de menciones del evento (`NumMentions`) |
| | `num_fuentes` | | Cantidad de fuentes que lo mencionan (`NumSources`) |
| | `num_articulos` | | Cantidad de artículos que lo mencionan (`NumArticles`) |
| | `tono_promedio` | | Tono promedio de la cobertura, de -100 a +100 (`AvgTone`) |
| | `fecha_actualizacion` | | Última actualización de los conteos |
| `ACTOR` | `actor_id` | PK | Identificador interno del actor (sustituta) |
| | `codigo_cameo` | UK | Código CAMEO completo del actor (`Actor1Code`) |
| | `nombre` | UK | Nombre del actor (`Actor1Name`) |
| | `pais_id` | FK | País del actor (`Actor1CountryCode`); opcional |
| | `grupo_conocido_cod` | | Organización conocida, p. ej. `UNO`, `NATO` (`Actor1KnownGroupCode`) |
| | `etnia_cod` | | Código étnico (`Actor1EthnicCode`) |
| | `religion1_cod`, `religion2_cod` | | Afiliación religiosa (`Actor1Religion1Code`, `Actor1Religion2Code`) |
| `UBICACION` | `ubicacion_id` | PK | Identificador interno de la ubicación (sustituta) |
| | `pais_id` | FK | País al que pertenece |
| | `feature_id` | UK | Identificador geográfico de GDELT (`*Geo_FeatureID`) |
| | `tipo_geo` | | Resolución: país, estado/provincia o ciudad (`*Geo_Type`) |
| | `nombre_completo` | | Nombre legible del lugar (`*Geo_FullName`) |
| | `adm1_cod`, `adm2_cod` | | División administrativa de 1.º y 2.º nivel (`*Geo_ADM1Code`, `*Geo_ADM2Code`) |
| | `latitud`, `longitud` | | Coordenadas (`*Geo_Lat`, `*Geo_Long`) |
| `FUENTE_MEDIO` | `fuente_id` | PK | Identificador interno del medio (sustituta) |
| | `dominio` | UK | Dominio del medio, p. ej. `bbc.co.uk` (`MentionSourceName`) |
| | `nombre` | | Nombre legible del medio |
| | `tipo_fuente` | | Tipo de fuente: web, broadcast, etc. (`MentionType`) |
| `CODIGO_CAMEO` | `cameo_cod` | PK | Código CAMEO de acción (`EventCode`, `EventBaseCode`, `EventRootCode`) |
| | `cameo_padre_cod` | FK | Código del nivel superior (recursiva); NULL en las raíces |
| | `descripcion` | | Descripción del código según el manual CAMEO |
| | `nivel` | | 1 = raíz, 2 = base, 3 = evento |
| | `quad_class` | | Clase general a la que pertenece el código |
| | `goldstein_base` | | Valor de referencia de la escala Goldstein |

**Tabla 3.** Atributos de las entidades de catálogo y asociativas.

| Entidad | Atributo | Llave | Descripción |
|---|---|---|---|
| `PAIS` | `pais_id` | PK | Identificador interno del país (sustituta) |
| | `codigo_fips` | UK | Código FIPS de 2 letras, usado en la geografía (`*Geo_CountryCode`) |
| | `codigo_cameo` | UK | Código CAMEO de 3 letras, usado en los actores (`Actor*CountryCode`) |
| | `nombre`, `region` | | Nombre del país y región geográfica |
| `TIPO_ACTOR` | `tipo_actor_cod` | PK | Código CAMEO del tipo de actor, p. ej. `GOV`, `NGO`, `BUS` |
| | `descripcion` | | Descripción del tipo |
| | `categoria` | | Agrupación para el análisis (Gobierno, ONG, Empresa, …) |
| `ACTOR_TIPO` | `actor_id` | PK, FK | Actor |
| | `orden` | PK | Posición del tipo: 1, 2 o 3 (`Actor1Type1Code` … `Actor1Type3Code`) |
| | `tipo_actor_cod` | FK | Tipo asignado |
| `EVENTO_ACTOR` | `evento_id` | PK, FK | Evento |
| | `rol` | PK | 1 = Actor1, 2 = Actor2 |
| | `actor_id` | FK | Actor que participa |
| | `ubicacion_id` | FK | Ubicación del actor en ese evento (`Actor1Geo_*`); opcional |
| `EVENTO_CATEGORIA` | `evento_id` | PK, FK | Evento |
| | `cameo_cod` | PK, FK | Código CAMEO asignado |
| | `es_principal` | | Si es el código principal (el `EventCode` de GDELT) |
| | `origen` | | Quién asignó el código: `GDELT` o `editor` |
| | `fecha_asignacion` | | Cuándo se asignó |
| `MENCION` | `mencion_id` | PK | Identificador interno de la mención (sustituta) |
| | `evento_id` | FK | Evento mencionado |
| | `fuente_id` | FK | Medio que publica la mención |
| | `url_articulo` | | URL del artículo (`MentionIdentifier`) |
| | `fecha_mencion` | | Fecha y hora de la mención (`MentionTimeDate`) |
| | `oracion_nro` | | Oración del artículo donde aparece el evento (`SentenceID`) |
| | `confianza` | | Confianza de la extracción, de 10 a 100 (`Confidence`) |
| | `largo_documento` | | Largo del artículo en caracteres (`MentionDocLen`) |
| | `tono_documento` | | Tono del artículo (`MentionDocTone`) |
| `ARTICULO_ORIGEN` | `evento_id` | PK, FK | Evento (PK compartida) |
| | `fuente_id` | FK | Medio que publicó el primer artículo |
| | `url` | | URL del primer artículo (`SOURCEURL`) |
| | `fecha_publicacion` | | Fecha de publicación del artículo |

### 1.3 Llaves

Una **llave natural** es un dato que ya existe fuera de la base de datos y que alcanza para identificar cada fila. Por ejemplo, `GlobalEventID` es el número que GDELT le asigna a cada evento. Una **llave sustituta** (*surrogate*) es un número que no significa nada por sí mismo y que el motor de base de datos genera automáticamente al insertar cada fila (por ejemplo, 1, 2, 3…). Solo sirve para identificar la fila dentro de la base.


**Tabla 4.** Llaves del modelo operativo.

| Tabla | Columna(s) | Tipo | Comentario |
|---|---|---|---|
| `NOTICIA_EVENTO` | `evento_id` | PK, sustituta | Secuencia interna |
| `NOTICIA_EVENTO` | `global_event_id` | UK, natural | Admite NULL (noticias propias del medio) |
| `NOTICIA_EVENTO` | `ubicacion_accion_id` | FK → `UBICACION` | Opcional |
| `ACTOR` | `actor_id` | PK, sustituta | GDELT no identifica a los actores |
| `ACTOR` | `codigo_cameo`, `nombre` | UK, natural compuesta | Un mismo código puede corresponder a varios nombres |
| `ACTOR` | `pais_id` | FK → `PAIS` | Opcional |
| `UBICACION` | `ubicacion_id` | PK, sustituta | |
| `UBICACION` | `feature_id` | UK, natural | Identificador geográfico de GDELT |
| `UBICACION` | `pais_id` | FK → `PAIS` | Obligatoria |
| `FUENTE_MEDIO` | `fuente_id` | PK, sustituta | |
| `FUENTE_MEDIO` | `dominio` | UK, natural | |
| `PAIS` | `pais_id` | PK, sustituta | |
| `PAIS` | `codigo_fips` / `codigo_cameo` | UK, naturales | Dos codificaciones distintas del mismo país |
| `CODIGO_CAMEO` | `cameo_cod` | PK, natural | |
| `CODIGO_CAMEO` | `cameo_padre_cod` | FK recursiva → `CODIGO_CAMEO` | NULL en los códigos raíz |
| `TIPO_ACTOR` | `tipo_actor_cod` | PK, natural | |
| `ACTOR_TIPO` | `actor_id`, `orden` | PK compuesta | `actor_id` también es FK → `ACTOR` |
| `ACTOR_TIPO` | `tipo_actor_cod` | FK → `TIPO_ACTOR` | |
| `EVENTO_ACTOR` | `evento_id`, `rol` | PK compuesta | `evento_id` también es FK → `NOTICIA_EVENTO` |
| `EVENTO_ACTOR` | `actor_id` / `ubicacion_id` | FK → `ACTOR` / FK → `UBICACION` | La ubicación es opcional |
| `EVENTO_CATEGORIA` | `evento_id`, `cameo_cod` | PK compuesta | Ambas columnas también son FK |
| `MENCION` | `mencion_id` | PK, sustituta | |
| `MENCION` | `evento_id` / `fuente_id` | FK → `NOTICIA_EVENTO` / FK → `FUENTE_MEDIO` | |
| `ARTICULO_ORIGEN` | `evento_id` | PK y FK → `NOTICIA_EVENTO` | Compartir la PK implementa la relación 1:1 |
| `ARTICULO_ORIGEN` | `fuente_id` | FK → `FUENTE_MEDIO` | |

**Llaves compuestas.**

- `EVENTO_ACTOR (evento_id, rol)`: un evento tiene como máximo dos actores (rol 1 o 2). Se usa el rol y no `actor_id` porque el mismo actor puede aparecer como Actor1 y Actor2 del mismo evento.
- `EVENTO_CATEGORIA (evento_id, cameo_cod)`: impide asignar dos veces el mismo código al mismo evento.
- `ACTOR_TIPO (actor_id, orden)`: conserva el orden de `Type1Code`, `Type2Code` y `Type3Code` y limita la cantidad a tres.

### 1.4 Relaciones

**Tabla 5.** Relaciones del modelo operativo.

| Entidad A | Entidad B | Card. | Particip. A | Particip. B | Implementación | Lectura |
|---|---|---|---|---|---|---|
| `NOTICIA_EVENTO` | `ARTICULO_ORIGEN` | **1:1** | Obligatoria | Obligatoria | PK compartida (`evento_id`) | Cada evento tiene exactamente un artículo de origen. |
| `NOTICIA_EVENTO` | `EVENTO_ACTOR` | 1:N | Opcional (0..2) | Obligatoria | FK `evento_id` | Un evento tiene entre 0 y 2 actores. |
| `ACTOR` | `EVENTO_ACTOR` | 1:N | Opcional | Obligatoria | FK `actor_id` | Un actor participa en 0 o más eventos. |
| `UBICACION` | `EVENTO_ACTOR` | 1:N | Opcional | Opcional | FK `ubicacion_id` (NULL) | La participación de un actor puede no estar geolocalizada. |
| `UBICACION` | `NOTICIA_EVENTO` | 1:N | Opcional | Opcional | FK `ubicacion_accion_id` (NULL) | Un evento ocurre en 0 o 1 lugar conocido. |
| `PAIS` | `UBICACION` | 1:N | Opcional | Obligatoria | FK `pais_id` | Toda ubicación pertenece a exactamente un país. |
| `PAIS` | `ACTOR` | 1:N | Opcional | Opcional | FK `pais_id` (NULL) | Un actor tiene 0 o 1 país (p. ej., una ONG transnacional no tiene). |
| `NOTICIA_EVENTO` | `EVENTO_CATEGORIA` | 1:N | Obligatoria (1..N) | Obligatoria | FK `evento_id` | Un evento tiene uno o más códigos CAMEO. |
| `CODIGO_CAMEO` | `EVENTO_CATEGORIA` | 1:N | Opcional | Obligatoria | FK `cameo_cod` | Un código clasifica a 0 o más eventos. |
| `CODIGO_CAMEO` | `CODIGO_CAMEO` | 1:N recursiva | Opcional | Opcional | FK `cameo_padre_cod` (NULL) | Un código tiene 0 o 1 padre: raíz → base → evento. |
| `NOTICIA_EVENTO` | `MENCION` | 1:N | Obligatoria (1..N) | Obligatoria | FK `evento_id` | Un evento tiene al menos una mención. |
| `FUENTE_MEDIO` | `MENCION` | 1:N | Opcional | Obligatoria | FK `fuente_id` | Toda mención la publica exactamente un medio. |
| `FUENTE_MEDIO` | `ARTICULO_ORIGEN` | 1:N | Opcional | Obligatoria | FK `fuente_id` | Un medio publicó 0 o más artículos de origen. |
| `ACTOR` | `ACTOR_TIPO` | 1:N | Opcional (0..3) | Obligatoria | FK `actor_id` | Un actor tiene entre 0 y 3 tipos. |
| `TIPO_ACTOR` | `ACTOR_TIPO` | 1:N | Opcional | Obligatoria | FK `tipo_actor_cod` | Un tipo se asigna a 0 o más actores. |

**Tabla 6.** Relaciones M:N, resueltas con tablas asociativas.

| Relación M:N | Tabla asociativa | Atributos de la relación |
|---|---|---|
| `NOTICIA_EVENTO` ↔ `ACTOR` | `EVENTO_ACTOR` | `rol`, `ubicacion_id` |
| `NOTICIA_EVENTO` ↔ `CODIGO_CAMEO` | `EVENTO_CATEGORIA` | `es_principal`, `origen`, `fecha_asignacion` |
| `NOTICIA_EVENTO` ↔ `FUENTE_MEDIO` | `MENCION` | URL, fecha, confianza y tono de cada mención |
| `ACTOR` ↔ `TIPO_ACTOR` | `ACTOR_TIPO` | `orden` |


## Actividad 2: Modelo dimensional (OLAP)

Referencia: `TP2.drawio`, **pestaña 2** (*Modelo dimensional (Estrella)*). Ahí se encuentran la tabla de hechos `FACT_COBERTURA_EVENTO` con sus métricas (`cantidad_eventos`, `num_menciones`, `num_fuentes`, `num_articulos`, `tono_promedio`, `tono_x_menciones`, `escala_goldstein`), las dimensiones asociadas (`DIM_TIEMPO`, `DIM_ACTOR`, `DIM_UBICACION`, `DIM_CAMEO`, `DIM_FUENTE`, `DIM_GRUPO_CATEGORIAS`) y la tabla puente `BRIDGE_EVENTO_CAMEO`. La dimensión `DIM_TIEMPO` incluye los atributos pedidos: `año`, `trimestre`, `mes`, `dia_semana`, `es_fin_de_semana` y `es_festivo`.

### Tabla puente

**Problema.** Una noticia puede clasificarse bajo varios códigos CAMEO. Entre el hecho y `DIM_CAMEO` hay entonces una relación **M:N**, que un esquema estrella no puede representar con una FK simple.

**Solución.** Tiene tres partes:

1. **Grupo de categorías.** Cada combinación distinta de códigos es una fila de `DIM_GRUPO_CATEGORIAS`. El hecho apunta a su grupo con una sola FK (`grupo_categorias_key`). Si varias noticias tienen la misma combinación, comparten el grupo, y la puente no crece con la cantidad de hechos. El atributo `es_multitematico` permite filtrar directamente los eventos con más de un código (pregunta 4).
2. **Tabla puente.** `BRIDGE_EVENTO_CAMEO (grupo_categorias_key, cameo_key)` lista los códigos de cada grupo, con PK compuesta. Cada columna es a la vez FK hacia su dimensión.
3. **Factor de ponderación.** Si un evento tiene *n* códigos y se agrupa por código, el evento aparece *n* veces y sus métricas se suman *n* veces (**doble conteo**). `factor_ponderacion = 1/n` reparte la métrica entre los códigos, de modo que el total vuelve a ser el del hecho.

El recorrido de una consulta es: `FACT_COBERTURA_EVENTO` → `DIM_GRUPO_CATEGORIAS` → `BRIDGE_EVENTO_CAMEO` → `DIM_CAMEO`.

**Tabla 7.** Ejemplo: una protesta con detenciones y acusaciones, clasificada con tres códigos.

| `grupo_categorias_key` | `cameo_key` (código) | `factor_ponderacion` | `es_principal` |
|---|---|---|---|
| 3 | 141 – Manifestarse o movilizarse | 0,3334 | TRUE |
| 3 | 173 – Arrestar, detener o acusar | 0,3333 | FALSE |
| 3 | 112 – Acusar | 0,3333 | FALSE |

Si el evento tiene 120 menciones, sumarlas por código sin ponderar daría 360; con el factor, la suma vuelve a ser 120. El código principal absorbe el redondeo para que los factores sumen exactamente 1. Además, el hecho conserva la FK directa `cameo_principal_key` para los análisis que usan un solo código por evento y no necesitan pasar por la puente.

## Actividad 3: Implementación en SQL

Todavía no se ha implementado.